In [1]:
import numpy as np
import pandas as pd

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src import sbm
from src import drc

# Data Import #

In [4]:
project_root = Path.cwd().parent
file_path = project_root / "data" / "sample_frtb_raw_data.xlsx"

sbm_data = pd.read_excel(file_path, sheet_name="sbm")
drc_data = pd.read_excel(file_path, sheet_name="drc")
rrao_data = pd.read_excel(file_path, sheet_name="rrao")

# [For edit] Data cleansing - SBM #

In [5]:
#Record reporting date and reporting currency
reporting_date = sbm_data.loc[0, "scenarioDate"]
reporting_currency = sbm_data.loc[0, "reportCurrency"]
extra_prescribed_currency_FX = None #for localized regulation which has extra currency that can enjoy lower risk weights

#Select only relevant columns for the analysis
relevant_col = ["Measure", "AssetClass", "SubClass", "CreditQuality", "Asset", "Ccy", "Curve", 
                "CorrelationVertex", "CorrelationClass", "Bucket", "RiskWeight", "PrimaryVertex", "ResidualVertex", 
                "Exposure", "StressScenarioUp", "StressScenarioDown", "UnderlyingPair", "DeliveryLocation"]
temp_sbm_data = sbm_data[relevant_col]
temp_sbm_data["CVR+/-"] = np.nan

#Clean sbm_data -- Risk Class Labels
def map_risk_class(row):
    if row["AssetClass"] == "EQ":
        return "Equity"
    elif row["AssetClass"] == "Cmdty":
        return "Commodity"
    elif row["AssetClass"] == "CSR":
        return str(np.select(
                            [
                            row["SubClass"] == "Secur",
                            row["SubClass"] == "SecCtp"
                            ], 
                            [
                            "CSR_sec",
                            "CSR_ctp"
                            ],
                            default="CSR_non"))
    else:
        return row["AssetClass"]

temp_sbm_data.insert(temp_sbm_data.columns.get_loc("AssetClass"), "RiskClass", temp_sbm_data.apply(map_risk_class, axis=1))

#Clean sbm_data -- GIRR curve type labels in SubClass
temp_sbm_data["SubClass"] = np.where(
                                temp_sbm_data["AssetClass"] == "GIRR", 
                                temp_sbm_data["SubClass"].map({
                                                            "RiskFree": "Yield", 
                                                            "CrossCurrency": "XCCY", 
                                                            "Inflation": "Inf"
                                                            }), 
                                temp_sbm_data["SubClass"]
                                )

#Clean sbm_data -- Curvature --> CVR under column "Measure"
temp_sbm_data["Measure"] = np.where(temp_sbm_data["Measure"] == "CVRE", "Curvature", temp_sbm_data["Measure"])

#Clean sbm_data -- Curvature exposure: In this sbm_dataset, CVR are calculated as CVR+(-) = StressScenarioUp(Down) - Exposure
def CVR_mapping(row):
    if row["Measure"] == "Curvature":
        if pd.notna(row["StressScenarioUp"]):
            row["Exposure"] = row["StressScenarioUp"] - row["Exposure"]
            row["CVR+/-"] = "CVR+"
        
        if pd.notna(row["StressScenarioDown"]):
            row["Exposure"] = row["StressScenarioDown"] - row["Exposure"]
            row["CVR+/-"] = "CVR-"

    return row

temp_sbm_data = temp_sbm_data.apply(CVR_mapping, axis=1)

/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/1165758130.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_sbm_data["CVR+/-"] = np.nan
/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/1165758130.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_sbm_data["SubClass"] = np.where(
/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/1165758130.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usin

# [For edit] Data cleansing - DRC #

In [6]:
#Check reporting date and reporting currency
if reporting_date != drc_data.loc[0, "scenarioDate"]:
    raise ValueError("Reporting date does not match between SBM and DRC dataset.")

if reporting_currency != drc_data.loc[0, "reportCurrency"]:
    raise ValueError("Reporting currency does not match between SBM and DRC dataset.")

#Select only relevant columns for the analysis
relevant_col = ["TradeId", "AssetClass", "Bucket", "Obligator", "SeniorityLevel", "Seniority", "CreditQuality",  
                "Instrument", "MaturityDate", "NotionalLong", "NotionalShort", "PLLong", "PLShort", "MarketValue"]
temp_drc_data = drc_data[relevant_col]

#Error checking
#Standardize and check for invalid AssetClass
temp_drc_data["AssetClass"] = np.select(
                                        [
                                        temp_drc_data["AssetClass"] == "NonSecuritization", 
                                        temp_drc_data["AssetClass"] == "Securitization", 
                                        temp_drc_data["AssetClass"] == "SecuritizationCTP"
                                        ], 
                                        [
                                        "DRC_non", 
                                        "DRC_sec", 
                                        "DRC_ctp"
                                        ], 
                                        default="Error"
                                        )
if any(temp_drc_data["AssetClass"] == "Error"):
    invalid_lines_asset_class = temp_drc_data[temp_drc_data["AssetClass"] == "Error"]
    invalid_trades = invalid_lines_asset_class["TradeId"].to_list()
    raise ValueError(f"Trade IDs {invalid_trades} have invalid DRC class.")

#Standardize and check for invalid Bucket
temp_drc_data["Bucket"] = np.select(
                                        [
                                        temp_drc_data["Bucket"] == "SOVEREIGN", 
                                        temp_drc_data["Bucket"] == "CORPORATE", 
                                        temp_drc_data["Bucket"] == "MUNICIPAL"
                                        ], 
                                        [
                                        "SOVEREIGN", 
                                        "CORPORATE", 
                                        "MUNICIPAL"
                                        ], 
                                        default="Error"
                                        )
if any(temp_drc_data["Bucket"] == "Error"):
    invalid_lines_asset_class = temp_drc_data[temp_drc_data["Bucket"] == "Error"]
    invalid_trades = invalid_lines_asset_class["TradeId"].to_list()
    raise ValueError(f"Trade IDs {invalid_trades} have invalid bucket.")

#Invalid maturity dates
if any((temp_drc_data["Instrument"] != "Equity") & (temp_drc_data["MaturityDate"].isna())):
    invalid_lines_maturity = temp_drc_data[(temp_drc_data["Instrument"] != "Equity") & (temp_drc_data["MaturityDate"].isna())]
    invalid_trades = invalid_lines_maturity["TradeId"].to_list()
    raise ValueError(f"Trade IDs {invalid_trades} have invalid maturity date.")

#Invalid credit quality
credit_quality_list = ["Zero", "AAA", "AA", "A", "BBB", "BB", "B", "CCC", "NR", "Defaulted"]
invalid_lines_credit_quality = temp_drc_data[~temp_drc_data["CreditQuality"].isin(credit_quality_list)]
if not invalid_lines_credit_quality.empty:
    invalid_trades = invalid_lines_credit_quality["TradeId"].to_list()
    raise ValueError(f"Trade IDs {invalid_trades} have invalid credit quality label.")

temp_drc_data[["NotionalLong", "NotionalShort", "PLLong", "PLShort", "MarketValue"]].fillna(0)

/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/4111058875.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_drc_data["AssetClass"] = np.select(
/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/4111058875.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_drc_data["Bucket"] = np.select(


,NotionalLong,NotionalShort,PLLong,PLShort,MarketValue
0,1.800000e+06,0.0,0.000000e+00,0.00000,1.800000e+06
1,4.700000e+06,0.0,1.244000e+07,0.00000,1.380930e+07
2,1.294920e+08,0.0,-1.144215e+04,0.00000,1.507052e+07
3,0.000000e+00,100000.0,0.000000e+00,-11442.14775,1.507052e+07
4,2.896000e+06,0.0,0.000000e+00,0.00000,2.896000e+06
5,3.237300e+06,0.0,2.073493e+08,0.00000,2.105866e+08
6,1.244000e+07,0.0,0.000000e+00,0.00000,1.244000e+07
7,0.000000e+00,6474600.0,0.000000e+00,0.00000,3.239442e+08
8,1.711000e+10,0.0,-9.882283e+08,0.00000,1.612177e+10
9,2.875000e+06,0.0,0.000000e+00,0.00000,2.875000e+06


# [For edit] Data cleansing - RRAO #

In [7]:
#Check reporting date and reporting currency
if reporting_date != rrao_data.loc[0, "scenarioDate"]:
    raise ValueError("Reporting date does not match between SBM and RRAO dataset.")

if reporting_currency != rrao_data.loc[0, "reportCurrency"]:
    raise ValueError("Reporting currency does not match between SBM and RRAO dataset.")

#Select only relevant columns for the analysis
relevant_col = ["TradeId", "Instrument", "Notional", "RRAOType"]
temp_rrao_data = rrao_data[relevant_col]

#Standardize RRAOType
temp_rrao_data["RRAOType"] = np.select  (
                                        [temp_rrao_data["RRAOType"] == "Exotic", temp_rrao_data["RRAOType"] == "Other"], 
                                        ["Exotic Underlying", "Other Residual Risk"],
                                        default="Error"
                                        )
if any(temp_rrao_data["RRAOType"] == "Error"):
    invalid_lines_rrao_type = temp_rrao_data[temp_rrao_data["RRAOType"] == "Error"]
    invalid_trades = invalid_lines_rrao_type["TradeId"].to_list()
    raise ValueError(f"Trade IDs {invalid_trades} have invalid RRAOType.")

/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/957665787.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_rrao_data["RRAOType"] = np.select  (


# [For edit] Transform to Standard Input Format #

In [8]:
#Create lists of relevant risk class to perform calculation
relevant_sbm_risk_class = temp_sbm_data["RiskClass"].drop_duplicates().reset_index(drop=True)
relevant_drc_risk_class = temp_drc_data["AssetClass"].drop_duplicates().reset_index(drop=True)

#Create dictionaries with all the risk charges under FRTB SA. If the risk class is irrelevant, user still gets an empty sheet as control.
data_SBM_class_map ={
                    "GIRR": None,
                    "CSR_non": None, 
                    "CSR_sec": None, 
                    "CSR_ctp": None, 
                    "Equity": None, 
                    "Commodity": None, 
                    "FX": None
                    }
data_DRC_class_map ={
                    "DRC_non": None, 
                    "DRC_sec": None, 
                    "DRC_ctp": None
                    }
data_RRAO_class_map={            
                    "RRAO": None
                    }

#SBM
for data_class in data_SBM_class_map:
    #Filter data only for the risk class
    filtered_data = temp_sbm_data[temp_sbm_data["RiskClass"] == data_class].reset_index(drop=True)
    
    if filtered_data.empty:
        #Empty dataframe
        df = pd.DataFrame()
    else:
        #Default columns
        df = pd.DataFrame({
                            "Sensi Type": filtered_data["Measure"],
                            "SA Bucket": filtered_data["Bucket"],
                            "CVR+/CVR-": filtered_data["CVR+/-"],
                            "Sensitivity (reporting currency equiv.)": filtered_data["Exposure"]
                            })
        loc = df.columns.get_loc("CVR+/CVR-")
        
        #Add columns according to specific dimensions of each risk class
        if data_class == "GIRR":
            df.insert(loc, "Curve Name", filtered_data["Curve"])
            df.insert(loc, "Curve Type", filtered_data["SubClass"])
            df.insert(loc, "Tenor", filtered_data["PrimaryVertex"])
            df.insert(loc, "Underlying Tenor", filtered_data["ResidualVertex"])
        elif data_class == "CSR_non" or data_class == "CSR_sec":
            df.insert(loc, "Issuer Name", filtered_data["Asset"])
            df.insert(loc, "Tenor", filtered_data["PrimaryVertex"])
            df.insert(loc, "Curve Type", filtered_data["CorrelationClass"])
            df.insert(loc, "Credit Rating (Optional)", filtered_data["CreditQuality"])
        elif data_class == "Equity":
            df.insert(loc, "Issuer", filtered_data["Asset"])
            df.insert(loc, "Price Type", filtered_data["SubClass"])
            df.insert(loc, "Tenor", filtered_data["PrimaryVertex"])
        elif data_class == "Commodity":
            df.insert(loc, "Commodity", filtered_data["Asset"])
            df.insert(loc, "Tenor", filtered_data["PrimaryVertex"])
            df.insert(loc, "Location", filtered_data["DeliveryLocation"])
        elif data_class == "FX":
            df.insert(loc, "Tenor", filtered_data["PrimaryVertex"])
            
    data_SBM_class_map[data_class] = df

#DRC    
for data_class in data_DRC_class_map:
    #Filter data only for the risk class
    filtered_data = temp_drc_data[temp_drc_data["AssetClass"] == data_class].reset_index(drop=True)
    
    if filtered_data.empty:
        #Empty dataframe
        df = pd.DataFrame()
    else:
        #Default columns
        df = pd.DataFrame({
                            "Bucket": filtered_data["Bucket"],
                            "Obligator": filtered_data["Obligator"],
                            "Credit Quality": filtered_data["CreditQuality"],
                            "Instrument": filtered_data["Instrument"], 
                            "Maturity": filtered_data["MaturityDate"]
                            })
        
        #Add columns according to specific dimensions of each risk class
        if data_class == "DRC_non":
            df["Seniority Level"] = filtered_data["SeniorityLevel"].astype(str)
            df["Credit Exposure"] = np.where(filtered_data["NotionalShort"].isna(), "Long", "Short")
            
            #Need adjustment according to dataset and product exposure
            df["Notional Amount"] = np.where(
                                            filtered_data["Instrument"] == "Equity",
                                            filtered_data["MarketValue"], 
                                            filtered_data[["NotionalLong", "NotionalShort"]].sum(axis=1)
                                            ) 
            df["P&L"] = np.where(
                                filtered_data["Instrument"] == "Equity",
                                0,
                                filtered_data[["PLLong","PLShort"]].sum(axis=1)
                                )
        else:
            raise ValueError(f"{data_class} is to be developed")
            
    data_DRC_class_map[data_class] = df

#RRAO
temp_rrao_data["Notional"] = np.abs(temp_rrao_data["Notional"]) 
data_RRAO_class_map["RRAO"] = temp_rrao_data.groupby(["RRAOType"])["Notional"].sum().reset_index()

/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/1806190324.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_rrao_data["Notional"] = np.abs(temp_rrao_data["Notional"])


# Export data to standardized format #

In [16]:
with pd.ExcelWriter(project_root / "outputs/sample_standardised_FRTB_data.xlsx") as writer:
    for keys, dfs in data_SBM_class_map.items():
        dfs.to_excel(writer, keys, index=False, freeze_panes=(1,0))
    
    for keys, dfs in data_DRC_class_map.items():
        dfs.to_excel(writer, keys, index=False, freeze_panes=(1,0))
        
    for keys, dfs in data_RRAO_class_map.items():
        dfs.to_excel(writer, keys, index=False, freeze_panes=(1,0))

/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/2654594224.py:3: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  dfs.to_excel(writer, keys, index=False, freeze_panes=(1,0))
/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/2654594224.py:6: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  dfs.to_excel(writer, keys, index=False, freeze_panes=(1,0))
/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/2654594224.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  dfs.to_excel(writer, keys, index=False, freeze_panes=(1,0))


# Calculate capital charges #

In [12]:
###SBM###
#1. Initiate appropriate risk class objects
# Map class names (strings) to the class objects
sbm_class_map = {
                "GIRR": sbm.GIRR,
                "CSR_non": sbm.CSR_non,
                "Equity": sbm.Equity,
                "Commodity": sbm.Commodity,
                "FX": sbm.FX
                }

sbm_capital_class_map ={
                        "GIRR": None,
                        "CSR_non": None, 
                        "CSR_sec": None, 
                        "CSR_ctp": None, 
                        "Equity": None, 
                        "Commodity": None, 
                        "FX": None, 
                        }

sbm_calculation_engines = {}
for name in relevant_sbm_risk_class:
    if name == "GIRR":
        sbm_calculation_engines[name] = sbm_class_map[name](data_SBM_class_map[name], reporting_currency)
    elif name == "CSR_sec":
        continue #placeholder
    elif name == "FX":
        sbm_calculation_engines[name] = sbm_class_map[name](data_SBM_class_map[name], extra_prescribed_currency_FX)
    else:
        sbm_calculation_engines[name] = sbm_class_map[name](data_SBM_class_map[name])
    
#2. Calculate capital charges
for risk_class, engine in sbm_calculation_engines.items():
    temp_df = pd.DataFrame()
    sensi_type = data_SBM_class_map[risk_class]["Sensi Type"].drop_duplicates()
    sensi_type = pd.Categorical(sensi_type, categories=["Delta", "Vega", "Curvature"], ordered=True)
    sensi_type = sensi_type.sort_values()
    for sensi in sensi_type:
        engine.weight_sensitivities()
        intra_result = engine.perform_intra_bucket_aggregation(sensi)
        capital_charges = engine.perform_across_bucket_aggregation(sensi, intra_result)
        temp_df = pd.concat([temp_df, capital_charges])
        
    sbm_capital_class_map[risk_class] = temp_df
    
#3. Save results
sbm_capital_class_map = pd.concat(sbm_capital_class_map).reset_index(names=["RiskClass","Sensitivity"])
sbm_capital_class_map

,RiskClass,Sensitivity,Medium,High,Low
0,GIRR,Delta,1.460009e+08,1.412439e+08,1.506146e+08
1,GIRR,Vega,5.550018e+03,5.589024e+03,5.510735e+03
2,GIRR,Curvature,5.590270e+05,5.593882e+05,5.586655e+05
3,CSR_non,Delta,4.200172e+03,4.200193e+03,4.200151e+03
4,CSR_non,Vega,3.014038e+03,3.012988e+03,3.015088e+03
5,CSR_non,Curvature,4.925952e+01,4.907265e+01,4.944568e+01
6,Equity,Delta,2.123322e+05,2.123460e+05,2.123184e+05
7,Equity,Vega,3.086858e+05,3.086573e+05,3.087143e+05
8,Equity,Curvature,9.869966e+04,9.851484e+04,9.888413e+04
9,Commodity,Delta,2.057770e+04,2.100000e+04,2.014654e+04


In [13]:
###SA-DRC###
#1. Initiate appropriate risk class objects
# Map class names (strings) to the class objects
drc_class_map = {
                "DRC_non": drc.DRC_non
                }

drc_capital_class_map ={
                        "DRC_non": None,
                        "DRC_sec": None, 
                        "DRC_ctp": None, 
                        }

drc_calculation_engines = {}
for name in relevant_drc_risk_class:
    drc_calculation_engines[name] = drc_class_map[name](data_DRC_class_map[name], reporting_date)

#2. Calculate capital charges
for risk_class, engine in drc_calculation_engines.items():
    engine.calculate_gross_JTD()
    engine.calculate_net_JTD()
    capital_charges = engine.calculate_DRC_charge()
    drc_capital_class_map[risk_class] = capital_charges
    
#3. Save results
drc_capital_class_map = pd.concat(drc_capital_class_map).reset_index(names=["RiskClass","Bucket"])
drc_capital_class_map

,RiskClass,Bucket,SA_DRC
0,DRC_non,CORPORATE,4.158772e+06
1,DRC_non,SOVEREIGN,1.935472e+06
2,DRC_non,MUNICIPAL,0.000000e+00


In [14]:
###RRAO###
rrao_risk_weights = {"Exotic Underlying": 0.1, "Other Residual Risk": 0.01}
temp_rrao_data["RRAO"] = temp_rrao_data["Notional"] * temp_rrao_data["RRAOType"].map(rrao_risk_weights)

rrao_capital_class_map = temp_rrao_data.groupby("RRAOType")["RRAO"].sum().reset_index()
rrao_capital_class_map

/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/1439677325.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_rrao_data["RRAO"] = temp_rrao_data["Notional"] * temp_rrao_data["RRAOType"].map(rrao_risk_weights)


,RRAOType,RRAO
0,Exotic Underlying,37653.80
1,Other Residual Risk,2012.33


In [17]:
#Export results
with pd.ExcelWriter(project_root / "outputs/sample_FRTB_capital_output.xlsx") as writer:
    
    sbm_capital_class_map.to_excel(writer, "SBM", index=False)
    drc_capital_class_map.to_excel(writer, "DRC", index=False)
    rrao_capital_class_map.to_excel(writer, "RRAO", index=False)

/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/2814109355.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  sbm_capital_class_map.to_excel(writer, "SBM", index=False)
/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/2814109355.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  drc_capital_class_map.to_excel(writer, "DRC", index=False)
/var/folders/lv/dczcf_d13xbg05qts66f6by00000gn/T/ipykernel_47150/2814109355.py:6: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  rrao_capital_class_map.to_excel(writer, "RRAO", index=False)
